# Fine-tune a small model for a narrow task (when prompting hits the cost wall)

A 5-way customer-support ticket classifier running on Claude Haiku at $1.50 per 1K calls has hit a $5K/month bill. The team wants to know if fine-tuning Llama 3.1 8B on Together AI ($0.20 per 1K) is worth the engineering effort. You have 150 minutes to generate 5K labelled examples with Claude Sonnet, fine-tune Llama 3.1 8B Instruct via Together's QLoRA fine-tune API, deploy the result behind Together's serverless inference, and benchmark cost-per-1K and accuracy against the Haiku baseline on a 500-example held-out test set.

The killer moment: after a 45 to 60 minute wall-clock fine-tune, the benchmark CSV shows your fine-tune within 2 points of Haiku at 7x cheaper, and you can defend the cost/accuracy trade with real numbers instead of hand-waving.

Estimated time: 150 minutes (the fine-tune itself is the wall-clock bottleneck; almost everything else is fast). Session window: 180 minutes. **Most expensive lab in the track. About eight bucks on a clean run.** The fine-tune itself is $5 to $8; the rest is synthetic data generation. If you start the fine-tune and then your kernel dies, the fine-tune job continues on Together's side; you do NOT have to pay again. Cleanup tears down the deployed model so inference stops billing.

Eight bucks. The fine-tune itself is most of the spend.


In [ ]:
# NBVAL_SKIP
# Pinned dependencies. labrun-checks stays on 0.7.0 per the rest of the track.
# together is the official Together AI SDK; anthropic powers the synthetic data
# generation and the Haiku baseline; boto3 puts the JSONL on S3; pandas and
# scikit-learn compute the benchmark accuracy.

!pip install --quiet labrun-checks==0.7.0 anthropic==0.42.0 together==1.3.13 boto3==1.35.92 pandas==2.2.3 scikit-learn==1.6.0


In [ ]:
# Imports and per-lab constants. Resource names use the lab slug prefix so
# the orphan scan can match by tag and by name.

import atexit
import getpass
import io
import json
import os
import random
import sys
import time
from collections import Counter

import boto3
from botocore.exceptions import ClientError
import pandas as pd

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupAdapter,
    CleanupResource,
)

LAB_ID = "fine-tune-small-model-cost-vs-prompting"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID

REGION = "us-east-1"
S3_BUCKET = f"labrun-{LAB_ID}-bucket"
TRAIN_KEY = "train.jsonl"
TEST_KEY = "test.jsonl"
BENCHMARK_CSV_PATH = "/content/benchmark_results.csv"

# Models. Pinned for reproducibility.
ANTHROPIC_SONNET_MODEL = "claude-sonnet-4-6"     # synthetic data generator
ANTHROPIC_HAIKU_MODEL = "claude-haiku-4-5"       # baseline classifier
TOGETHER_BASE_MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct-Reference"

# Pricing constants. Verified against Anthropic + Together public pricing as
# of early 2026. Per 1M tokens.
HAIKU_INPUT_USD_PER_M = 0.80
HAIKU_OUTPUT_USD_PER_M = 4.00
TOGETHER_LLAMA_INPUT_USD_PER_M = 0.20
TOGETHER_LLAMA_OUTPUT_USD_PER_M = 0.20

# Classification task: 5 customer-support ticket categories.
LABELS = ["billing", "account", "technical", "shipping", "refund"]
TARGET_TRAIN_ROWS = 5000
TARGET_TEST_ROWS = 500

# Fine-tune naming so the Together SDK can find a prior job on re-entry. The
# Together SDK supports a free-text `suffix`; we encode the lab slug so a
# kernel-disconnect resume can locate the in-flight job.
TOGETHER_SUFFIX = f"labrun-{LAB_ID}".replace("_", "-")[:40]

POLL_CEILING_SECONDS = 90 * 60  # 90 minutes per CP2 ceiling.


In [ ]:
# NBVAL_SKIP
# BYOK setup. Four secrets via getpass: the labrun session token,
# ANTHROPIC_API_KEY, TOGETHER_API_KEY, AWS_ACCESS_KEY_ID + SECRET_ACCESS_KEY.
# Then ping Anthropic + Together to validate the keys cheaply, ping AWS STS
# to confirm credentials, and register the labrun session.

import anthropic
import together

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
ANTHROPIC_API_KEY = getpass.getpass("ANTHROPIC_API_KEY: ").strip()
TOGETHER_API_KEY = getpass.getpass("TOGETHER_API_KEY: ").strip()
AWS_ACCESS_KEY_ID = getpass.getpass("AWS_ACCESS_KEY_ID: ").strip()
AWS_SECRET_ACCESS_KEY = getpass.getpass("AWS_SECRET_ACCESS_KEY: ").strip()

if not all([ANTHROPIC_API_KEY, TOGETHER_API_KEY, AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY]):
    print("All four credentials are required. Re-run this cell.")
    raise SystemExit(1)

os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY
os.environ["TOGETHER_API_KEY"] = TOGETHER_API_KEY
os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
os.environ["AWS_DEFAULT_REGION"] = REGION

# Anthropic ping. One Haiku call costs a fraction of a cent.
try:
    anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    _ping = anthropic_client.messages.create(
        model=ANTHROPIC_HAIKU_MODEL,
        max_tokens=8,
        messages=[{"role": "user", "content": "reply with the single word: ok"}],
    )
    print(f"Anthropic key validated. Haiku replied: {_ping.content[0].text!r}")
except Exception as exc:
    print(f"Anthropic key rejected: {exc!r}")
    print("Refresh ANTHROPIC_API_KEY and re-run this cell.")
    raise SystemExit(1)

# Together ping. The cheapest call is a list-models call; no token spend.
together_client = together.Together(api_key=TOGETHER_API_KEY)
try:
    _models = list(together_client.models.list())
    print(f"Together key validated. {len(_models)} models visible to this key.")
except Exception as exc:
    print(f"Together key rejected: {exc!r}")
    print("Refresh TOGETHER_API_KEY and re-run this cell.")
    raise SystemExit(1)

# AWS STS ping. Confirms the credentials and prints the account id.
try:
    sts = boto3.client(
        "sts",
        region_name=REGION,
        aws_access_key_id=AWS_ACCESS_KEY_ID,
        aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    )
    _ident = sts.get_caller_identity()
    print(f"AWS credentials validated. Account: {_ident['Account']}. Region: {REGION}.")
except Exception as exc:
    print(f"AWS credentials rejected: {exc!r}")
    print("Refresh AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY and re-run this cell.")
    raise SystemExit(1)

CREDENTIALS = {
    "anthropic_api_key": ANTHROPIC_API_KEY,
    "together_api_key": TOGETHER_API_KEY,
    "aws_access_key_id": AWS_ACCESS_KEY_ID,
    "aws_secret_access_key": AWS_SECRET_ACCESS_KEY,
    "region": REGION,
}

register(session_token=session_token, lab_id=LAB_ID, credentials=CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")


In [ ]:
# NBVAL_SKIP
# Cleanup manifest, custom FineTuneCleanupAdapter, atexit safety net, and
# orphan scan. The Together fine-tune job is the only "critical while running"
# resource: charges accrue from job start until the job reports a terminal
# state. Cleanup cancels the job first if it is still active.
# TODO: migrate to together.py adapter once shipped (track brief Section 4).

CLEANUP_MANIFEST = [
    CleanupResource(
        type="together_fine_tune_job",
        id="placeholder-set-at-submit-time",
        region="together",
        critical=True,
        cli_delete_command="together fine-tuning cancel <job_id>",
    ),
    CleanupResource(
        type="together_model",
        id="placeholder-set-after-completion",
        region="together",
        cli_delete_command="together models delete <model_id>",
    ),
    CleanupResource(
        type="s3_bucket",
        id=S3_BUCKET,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{S3_BUCKET} --force",
    ),
    CleanupResource(
        type="local_file",
        id=BENCHMARK_CSV_PATH,
        region="local",
        cli_delete_command=f"rm -f {BENCHMARK_CSV_PATH}",
    ),
]


class FineTuneCleanupAdapter:
    """Cleans up Together AI fine-tune artifacts, the S3 training bucket, and the
    local benchmark CSV. TODO: migrate to together.py adapter once shipped.

    Branches by resource.type:
      - together_fine_tune_job: cancel if non-terminal, then ignore (job
        metadata persists on Together's side but costs nothing at rest).
      - together_model: delete the fine-tuned model.
      - s3_bucket: empty + delete the bucket.
      - local_file: rm -f.
    """

    _TERMINAL_STATES = {"completed", "cancelled", "error", "failed", "user_error"}

    def __init__(self, together_client, s3_client):
        self._together = together_client
        self._s3 = s3_client

    def _ft_status(self, job_id):
        try:
            job = self._together.fine_tuning.retrieve(job_id)
        except Exception:
            return None
        status = getattr(job, "status", None)
        if status is None and isinstance(job, dict):
            status = job.get("status")
        return getattr(status, "value", status)

    def delete_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id

        if rtype == "together_fine_tune_job":
            if rid.startswith("placeholder"):
                return
            status = self._ft_status(rid)
            if status is None:
                return
            if status not in self._TERMINAL_STATES:
                try:
                    self._together.fine_tuning.cancel(rid)
                except Exception as exc:
                    raise RuntimeError(f"failed to cancel fine-tune {rid!r}: {exc!r}")
            return

        if rtype == "together_model":
            if rid.startswith("placeholder"):
                return
            try:
                self._together.models.delete(rid)
            except Exception as exc:
                msg = str(exc).lower()
                if "not found" in msg or "404" in msg:
                    return
                raise

        if rtype == "s3_bucket":
            try:
                paginator = self._s3.get_paginator("list_object_versions")
                for page in paginator.paginate(Bucket=rid):
                    for kind in ("Versions", "DeleteMarkers"):
                        for obj in page.get(kind, []) or []:
                            self._s3.delete_object(
                                Bucket=rid,
                                Key=obj["Key"],
                                VersionId=obj["VersionId"],
                            )
            except ClientError as exc:
                ecode = exc.response.get("Error", {}).get("Code")
                if ecode in ("NoSuchBucket",):
                    return
                if ecode not in ("NoSuchVersioning", "InvalidArgument"):
                    raise
            try:
                paginator = self._s3.get_paginator("list_objects_v2")
                for page in paginator.paginate(Bucket=rid):
                    for obj in page.get("Contents", []) or []:
                        self._s3.delete_object(Bucket=rid, Key=obj["Key"])
            except ClientError as exc:
                ecode = exc.response.get("Error", {}).get("Code")
                if ecode not in ("NoSuchBucket",):
                    raise
            try:
                self._s3.delete_bucket(Bucket=rid)
            except ClientError as exc:
                ecode = exc.response.get("Error", {}).get("Code")
                if ecode in ("NoSuchBucket",):
                    return
                raise

        if rtype == "local_file":
            try:
                os.remove(rid)
            except FileNotFoundError:
                pass

    def describe_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype == "together_fine_tune_job":
            if rid.startswith("placeholder"):
                return False
            status = self._ft_status(rid)
            if status is None:
                return False
            return status not in self._TERMINAL_STATES
        if rtype == "together_model":
            if rid.startswith("placeholder"):
                return False
            try:
                for m in self._together.models.list():
                    model_id = getattr(m, "id", None) or (m.get("id") if isinstance(m, dict) else None)
                    if model_id == rid:
                        return True
                return False
            except Exception:
                return False
        if rtype == "s3_bucket":
            try:
                self._s3.head_bucket(Bucket=rid)
                return True
            except ClientError:
                return False
        if rtype == "local_file":
            return os.path.exists(rid)
        return False

    def tag_scan(self, credentials, lab_slug, region):
        # Two surfaces to scan: S3 buckets (by name prefix) and Together
        # fine-tune jobs (by `suffix` prefix). Together's free-form suffix
        # is the closest analogue to an AWS tag for this provider.
        arns: list[str] = []
        prefix = f"labrun-{lab_slug}"
        try:
            buckets = self._s3.list_buckets().get("Buckets", []) or []
            for b in buckets:
                if b["Name"].startswith(prefix):
                    arns.append(f"arn:aws:s3:::{b['Name']}")
        except Exception:
            pass
        try:
            for job in self._together.fine_tuning.list():
                suffix = getattr(job, "suffix", None) or (job.get("suffix") if isinstance(job, dict) else None)
                status = getattr(job, "status", None) or (job.get("status") if isinstance(job, dict) else None)
                status = getattr(status, "value", status)
                if not suffix or not suffix.startswith(prefix):
                    continue
                if status in self._TERMINAL_STATES:
                    continue
                job_id = getattr(job, "id", None) or (job.get("id") if isinstance(job, dict) else None)
                if job_id:
                    arns.append(f"together:fine-tune-job/{job_id}")
        except Exception:
            pass
        return arns


s3_client = boto3.client(
    "s3",
    region_name=REGION,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
)

CLEANUP_ADAPTER = FineTuneCleanupAdapter(together_client=together_client, s3_client=s3_client)


def _atexit_cleanup():
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


# Re-entry scan. If a prior session left a Together fine-tune job tagged
# with this lab's suffix in a non-terminal state, surface it and adopt it
# (so a kernel disconnect during the 45 to 60 min wait does NOT double-charge
# the student). The S3 bucket from a prior session blocks execution and
# routes the student to cleanup.
existing_job_id = None
existing_job_status = None
try:
    for job in together_client.fine_tuning.list():
        suffix = getattr(job, "suffix", None) or (job.get("suffix") if isinstance(job, dict) else None)
        status = getattr(job, "status", None) or (job.get("status") if isinstance(job, dict) else None)
        status = getattr(status, "value", status)
        job_id = getattr(job, "id", None) or (job.get("id") if isinstance(job, dict) else None)
        if suffix == TOGETHER_SUFFIX and status not in FineTuneCleanupAdapter._TERMINAL_STATES:
            existing_job_id = job_id
            existing_job_status = status
            break
except Exception as exc:
    print(f"WARNING: could not list Together fine-tune jobs ({exc!r}). Continuing.")

try:
    s3_client.head_bucket(Bucket=S3_BUCKET)
    bucket_exists = True
except ClientError as _exc:
    bucket_exists = False
    err_code = _exc.response.get("Error", {}).get("Code")
    if err_code == "403":
        print(f"S3 bucket name {S3_BUCKET!r} is taken by another account. Pick a different lab slug or contact support.")
        raise SystemExit(1)

if existing_job_id:
    print(f"RESUME DETECTED: prior fine-tune job {existing_job_id!r} (status={existing_job_status!r}) is tagged with this lab's suffix.")
    print("Setting it as the active job for Task 2; no double charge.")
    CLEANUP_MANIFEST[0].id = existing_job_id

if bucket_exists and not existing_job_id:
    print(f"BLOCKED: S3 bucket {S3_BUCKET!r} already exists from a prior session, but no active Together job is associated.")
    print("Run the cleanup cell at the bottom first, then re-run setup.")
    raise SystemExit(1)

# Create the S3 bucket if it does not already exist.
if not bucket_exists:
    try:
        if REGION == "us-east-1":
            s3_client.create_bucket(Bucket=S3_BUCKET)
        else:
            s3_client.create_bucket(
                Bucket=S3_BUCKET,
                CreateBucketConfiguration={"LocationConstraint": REGION},
            )
        s3_client.put_bucket_tagging(
            Bucket=S3_BUCKET,
            Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
        )
        print(f"Created S3 bucket {S3_BUCKET}.")
    except ClientError as exc:
        ecode = exc.response.get("Error", {}).get("Code")
        if ecode in ("BucketAlreadyOwnedByYou",):
            print(f"S3 bucket {S3_BUCKET} already owned by this account; continuing.")
        else:
            print(f"FAILED to create S3 bucket: {exc!r}")
            raise SystemExit(1)

print(f"Setup complete. Together suffix tag for this run: {TOGETHER_SUFFIX}.")


## Task 1: Synthesize a balanced 5K training set with Claude Sonnet, upload to S3

You need 5K training rows AND 500 test rows. Five customer-support ticket categories (`billing`, `account`, `technical`, `shipping`, `refund`). Sample labels from a **uniform** distribution before you call Sonnet; do not let the LLM pick the class. Two design points the brief calls out:

1. Together's fine-tune API requires JSONL with the chat-completions schema: `{"messages": [{"role": "user", "content": "<ticket text>"}, {"role": "assistant", "content": "<label>"}]}`. Using the older `{"text": "..."}` schema produces a checkpoint failure on submit.
2. Class balance is checked at CP1. If one class drifts more than 10 percentage points off the uniform target (`1/5 = 20%`), the checkpoint fails. Uniform label sampling solves both problems for free.

Generate, write the rows in memory, then upload `train.jsonl` and `test.jsonl` to `s3://labrun-fine-tune-small-model-cost-vs-prompting-bucket/`. Sonnet spend for 5500 short generations is roughly $0.90.


In [ ]:
# NBVAL_SKIP
# Task 1: synthesize and upload 5500 labelled examples (5000 train + 500 test).
#
# The structural scaffolding (label sampling, Sonnet prompt template, JSONL
# writer, S3 upload) is wired up. You fill in the two boto3-like calls that
# actually do the work:
#   - the Sonnet completion call inside _generate_one_example()
#   - the two S3 put_object calls at the bottom

random.seed(42)


def _sample_labels(n):
    """Uniform-sample labels so no class is favoured at generation time."""
    return [random.choice(LABELS) for _ in range(n)]


TICKET_PROMPT_TEMPLATE = (
    "You are a customer-support-ticket synthesizer. Write ONE realistic, "
    "1- to 3-sentence customer support ticket for the category {label!r}. "
    "Do not include the label inside the ticket text. Do not include any "
    "preamble. Output only the ticket body.\n\n"
    "Category cheat sheet:\n"
    "- billing: invoices, charges, refunds-disputed, payment methods\n"
    "- account: login, MFA, password reset, profile, deletion\n"
    "- technical: bugs, errors, broken features, app crashes\n"
    "- shipping: delivery status, lost packages, tracking, address\n"
    "- refund: return requests, refund status, return labels"
)


def _generate_one_example(label):
    """Call Claude Sonnet to synthesize one ticket for the given label.

    Returns a dict in Together's chat-completions JSONL shape:
        {"messages": [
            {"role": "user", "content": "<ticket text>"},
            {"role": "assistant", "content": "<label>"},
        ]}
    """
    prompt = TICKET_PROMPT_TEMPLATE.format(label=label)
    # YOUR CODE: call anthropic_client.messages.create with model
    # ANTHROPIC_SONNET_MODEL, max_tokens 120, and the prompt above as a single
    # user message. Capture the text in `ticket_text`.

    return {
        "messages": [
            {"role": "user", "content": ticket_text.strip()},
            {"role": "assistant", "content": label},
        ]
    }


def _generate_set(n, set_name):
    labels = _sample_labels(n)
    rows = []
    last_print = time.time()
    for i, lab in enumerate(labels):
        try:
            rows.append(_generate_one_example(lab))
        except Exception as exc:
            print(f"WARNING: generation {i} for {lab!r} failed ({exc!r}); retrying once after 2s")
            time.sleep(2)
            rows.append(_generate_one_example(lab))
        if time.time() - last_print > 10:
            print(f"  {set_name}: {i + 1}/{n} generated")
            last_print = time.time()
    return rows


print("Generating training set (5000 rows). Expect roughly 12 to 18 minutes wall-clock.")
train_rows = _generate_set(TARGET_TRAIN_ROWS, "train")
print(f"Training set ready: {len(train_rows)} rows.")
print(f"Class distribution: {Counter(r['messages'][1]['content'] for r in train_rows)}")

print("\nGenerating test set (500 rows).")
test_rows = _generate_set(TARGET_TEST_ROWS, "test")
print(f"Test set ready: {len(test_rows)} rows.")
print(f"Class distribution: {Counter(r['messages'][1]['content'] for r in test_rows)}")


def _to_jsonl_bytes(rows):
    buf = io.StringIO()
    for r in rows:
        buf.write(json.dumps(r))
        buf.write("\n")
    return buf.getvalue().encode("utf-8")


train_bytes = _to_jsonl_bytes(train_rows)
test_bytes = _to_jsonl_bytes(test_rows)

# YOUR CODE: upload train_bytes to s3_client at Bucket=S3_BUCKET, Key=TRAIN_KEY.

# YOUR CODE: upload test_bytes to s3_client at Bucket=S3_BUCKET, Key=TEST_KEY.

print(f"\nUploaded train.jsonl ({len(train_bytes):,} bytes) and test.jsonl ({len(test_bytes):,} bytes) to s3://{S3_BUCKET}/")


In [ ]:
# NBVAL_SKIP
# Checkpoint 1: train.jsonl exists in S3, has 5000 rows in the Together
# chat-completions schema, and label distribution is balanced within 10
# percentage points of uniform (each class between 10% and 30%).


def cp1_validator(session):
    creds = session.credentials
    s3 = boto3.client(
        "s3",
        region_name=creds["region"],
        aws_access_key_id=creds["aws_access_key_id"],
        aws_secret_access_key=creds["aws_secret_access_key"],
    )
    try:
        obj = s3.get_object(Bucket=S3_BUCKET, Key=TRAIN_KEY)
    except ClientError as exc:
        return CheckpointResult(
            "fail",
            error_reason=f"train.jsonl not found in s3://{S3_BUCKET}/{TRAIN_KEY} ({exc.response.get('Error', {}).get('Code', 'unknown')})",
        )

    body = obj["Body"].read().decode("utf-8")
    rows = [json.loads(line) for line in body.splitlines() if line.strip()]
    if len(rows) != TARGET_TRAIN_ROWS:
        return CheckpointResult(
            "fail",
            error_reason=f"train.jsonl has {len(rows)} rows; expected {TARGET_TRAIN_ROWS}",
        )

    # Schema check. Each row must be {"messages": [user, assistant]}.
    for i, r in enumerate(rows[:5]):
        if "messages" not in r or len(r["messages"]) != 2:
            return CheckpointResult(
                "fail",
                error_reason=(
                    f"row {i} does not have the Together chat-completions shape; "
                    f"expected {{'messages': [user, assistant]}}, got keys {list(r.keys())!r}"
                ),
            )

    counts = Counter(r["messages"][1]["content"] for r in rows)
    if set(counts.keys()) != set(LABELS):
        return CheckpointResult(
            "fail",
            error_reason=f"labels in train.jsonl ({sorted(counts.keys())}) do not match expected ({sorted(LABELS)})",
        )

    for lab, n in counts.items():
        share = n / len(rows)
        if share < 0.10 or share > 0.30:
            return CheckpointResult(
                "fail",
                error_reason=(
                    f"class {lab!r} has share {share:.1%}; expected within 10 points of 20%. "
                    f"Counts: {dict(counts)}"
                ),
            )

    return CheckpointResult("pass")


check(1, cp1_validator)


<details><summary>Hint 1 (nudge)</summary>

Two checks fire here: row count AND class balance. Sample your labels FIRST from a uniform distribution, then ask Sonnet to write a ticket for the pre-chosen label. If you let Sonnet pick the category, the model has its own priors and the distribution drifts.

</details>

<details><summary>Hint 2 (direction)</summary>

The Together fine-tune API expects each JSONL line in this exact shape:

```
{"messages": [{"role": "user", "content": "<ticket text>"}, {"role": "assistant", "content": "<label>"}]}
```

The legacy `{"text": "<prompt><sep><completion>"}` shape is rejected at job submit time. Confirm your JSONL contains `messages` keys, not `text`.

S3 uploads from in-memory bytes use `s3_client.put_object(Bucket=..., Key=..., Body=<bytes>)`. No special encoding flag is needed.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Full working Sonnet call and S3 uploads:

```python
def _generate_one_example(label):
    prompt = TICKET_PROMPT_TEMPLATE.format(label=label)
    resp = anthropic_client.messages.create(
        model=ANTHROPIC_SONNET_MODEL,
        max_tokens=120,
        messages=[{"role": "user", "content": prompt}],
    )
    ticket_text = resp.content[0].text
    return {
        "messages": [
            {"role": "user", "content": ticket_text.strip()},
            {"role": "assistant", "content": label},
        ]
    }

s3_client.put_object(Bucket=S3_BUCKET, Key=TRAIN_KEY, Body=train_bytes)
s3_client.put_object(Bucket=S3_BUCKET, Key=TEST_KEY, Body=test_bytes)
```

</details>

**Wallet check.** 5500 short Sonnet generations at roughly 150 input + 80 output tokens each. Total tokens around 825K input + 440K output. Sonnet pricing: $3/$15 per 1M. Damage so far: roughly $0.83 (input) + $0.07 (output) plus the cost of S3 PUT requests (a fraction of a cent). About 90 cents. Your morning coffee cost five times more.


## Task 2: Submit a Together fine-tune job and wait for `completed`

Together's fine-tune API uploads the training file, kicks off a QLoRA fine-tune on `meta-llama/Meta-Llama-3.1-8B-Instruct-Reference`, and returns a job id. The job runs 45 to 60 minutes wall-clock for 5K rows. **Cost: $5 to $8 for the full run. This is the only spend you cannot interrupt cheaply.**

If your Colab kernel disconnects during the wait, the Together job keeps running on Together's side. When you reconnect, re-run setup; the re-entry scan adopts the in-flight job and records the existing job id. Then re-run this task cell, which sees the existing job id and resumes polling. No double charge.

Tag the job with `suffix=TOGETHER_SUFFIX` so the resume path can find it. Use `n_epochs=3`, `learning_rate=1e-4`, the QLoRA training type. Poll status every 30 seconds with a 90-minute ceiling.


In [ ]:
# NBVAL_SKIP
# Task 2: submit the fine-tune, poll until completed, capture the resulting
# fine-tuned model id.

from together.types.finetune import FinetuneLoraTrainingType


def _job_status(job_id):
    job = together_client.fine_tuning.retrieve(job_id)
    status = getattr(job, "status", None) or (job.get("status") if isinstance(job, dict) else None)
    return getattr(status, "value", status), job


def _job_output_model(job):
    # The completed job exposes the fine-tuned model name. The SDK has called
    # this `output_name` in some versions and `model_output_name` in others;
    # accept either.
    for attr in ("output_name", "model_output_name", "fine_tuned_model"):
        v = getattr(job, attr, None)
        if v:
            return v
    if isinstance(job, dict):
        for k in ("output_name", "model_output_name", "fine_tuned_model"):
            if job.get(k):
                return job[k]
    return None


# Step 1: upload the train.jsonl to Together's file store (Together's
# fine-tune API consumes Together-hosted files, not direct S3 urls).
print("Uploading train.jsonl to Together file store.")
try:
    s3_obj = s3_client.get_object(Bucket=S3_BUCKET, Key=TRAIN_KEY)
    local_train_path = "/content/train.jsonl"
    with open(local_train_path, "wb") as f:
        f.write(s3_obj["Body"].read())
    together_file = together_client.files.upload(file=local_train_path, check=True)
    train_file_id = getattr(together_file, "id", None) or together_file.get("id")
    print(f"Together file id: {train_file_id}")
except Exception as exc:
    print(f"FAILED to upload train.jsonl to Together: {exc!r}")
    raise SystemExit(1)

# Step 2: submit (or adopt) the fine-tune job.
if CLEANUP_MANIFEST[0].id.startswith("placeholder"):
    print("Submitting new fine-tune job (Llama 3.1 8B Instruct QLoRA, 3 epochs).")
    # YOUR CODE: call together_client.fine_tuning.create with the training
    # file id, model TOGETHER_BASE_MODEL, n_epochs=3, learning_rate=1e-4,
    # a QLoRA training type (FinetuneLoraTrainingType()), and
    # suffix=TOGETHER_SUFFIX. Capture the return value in `job`.

    job_id = getattr(job, "id", None) or job.get("id")
    CLEANUP_MANIFEST[0].id = job_id
    print(f"Submitted job: {job_id}")
else:
    job_id = CLEANUP_MANIFEST[0].id
    print(f"Adopting in-flight job from prior session: {job_id}")

# Step 3: poll. Together fine-tunes typically take 45 to 60 minutes for 5K
# rows; the ceiling is 90 minutes.
print("Polling status every 30s. The fine-tune is the wall-clock bottleneck.")
deadline = time.time() + POLL_CEILING_SECONDS
final_status = None
final_job = None
while time.time() < deadline:
    try:
        status, j = _job_status(job_id)
    except Exception as exc:
        print(f"  retrieve error ({exc!r}); retrying in 30s")
        time.sleep(30)
        continue
    elapsed = int(POLL_CEILING_SECONDS - (deadline - time.time()))
    print(f"  t+{elapsed:>5}s status={status!r}")
    if status in ("completed", "error", "failed", "cancelled", "user_error"):
        final_status = status
        final_job = j
        break
    time.sleep(30)

if final_status != "completed":
    print(f"Fine-tune did not reach completed (final={final_status!r}). Re-run after fixing the issue.")
else:
    output_model = _job_output_model(final_job)
    print(f"\nFine-tune complete. Output model id: {output_model!r}")
    if output_model:
        CLEANUP_MANIFEST[1].id = output_model


In [ ]:
# NBVAL_SKIP
# Checkpoint 2: the lab's tagged fine-tune job is in status='completed'.


def cp2_validator(session):
    job_id = CLEANUP_MANIFEST[0].id
    if not job_id or job_id.startswith("placeholder"):
        return CheckpointResult(
            "fail",
            error_reason="no fine-tune job id recorded; submit the job first",
        )
    try:
        job = together_client.fine_tuning.retrieve(job_id)
    except Exception as exc:
        return CheckpointResult(
            "error",
            error_reason=f"could not retrieve Together job {job_id!r}: {exc!r}",
        )
    status = getattr(job, "status", None) or (job.get("status") if isinstance(job, dict) else None)
    status = getattr(status, "value", status)
    if status != "completed":
        return CheckpointResult(
            "fail",
            error_reason=(
                f"fine-tune job {job_id!r} is in status {status!r}; expected 'completed'. "
                f"If still running, wait and re-run this cell."
            ),
        )

    # Backfill the output model id into the cleanup manifest so later
    # checkpoints and cleanup can reach it even if the user re-ran the cell.
    if CLEANUP_MANIFEST[1].id.startswith("placeholder"):
        for attr in ("output_name", "model_output_name", "fine_tuned_model"):
            v = getattr(job, attr, None)
            if v:
                CLEANUP_MANIFEST[1].id = v
                break

    return CheckpointResult("pass")


check(2, cp2_validator)


<details><summary>Hint 1 (nudge)</summary>

If the job rejected at submit, the JSONL schema is the usual culprit. Re-read Hint 2 of Task 1: Together expects `{"messages": [...]}`, not `{"text": "..."}`. If the job is still polling, you are not done; wait it out.

</details>

<details><summary>Hint 2 (direction)</summary>

Together's QLoRA training type is the cheap option and matches the brief. The SDK exposes it as `FinetuneLoraTrainingType()` from `together.types.finetune`. The `suffix` parameter is free text; using the lab slug means the orphan scan and the re-entry resume path both find your job by tag.

The job lifecycle: `pending` -> `queued` -> `running` -> `completed`. Other terminal states (`error`, `failed`, `cancelled`, `user_error`) are failure paths and surface in the polling output. Treat anything non-terminal as still-running.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Full working submit:

```python
job = together_client.fine_tuning.create(
    training_file=train_file_id,
    model=TOGETHER_BASE_MODEL,
    n_epochs=3,
    learning_rate=1e-4,
    training_type=FinetuneLoraTrainingType(),
    suffix=TOGETHER_SUFFIX,
)
```

</details>

**Wallet check.** Fine-tune done. Together bill: about $6.50 (QLoRA on 5K rows, 3 epochs, 8B model). Running total: roughly $7.40. Inference benchmarking is next; that is pennies.


## Task 3: Call the fine-tuned model via Together serverless

The fine-tune is done. Together's serverless inference endpoint hosts the model with no idle billing; you pay per request. **The first call after a long idle period cold-starts for 30 to 60 seconds. Retry with exponential backoff so a transient cold-start does not look like a real error.**

Build one helper, `classify_with_finetune(text)`, that returns the predicted label. Use the chat-completions API: a single user message containing the ticket, `max_tokens=8` because the response is a single category word, `temperature=0` for determinism.


In [ ]:
# NBVAL_SKIP
# Task 3: call the fine-tuned model. Retry with backoff on transient errors
# so a cold-start does not surface as a real failure.

FINETUNE_MODEL_ID = CLEANUP_MANIFEST[1].id
if FINETUNE_MODEL_ID.startswith("placeholder"):
    print("ERROR: no fine-tuned model id captured. Re-run Task 2 and confirm it reported 'output model id'.")
    raise SystemExit(1)


def classify_with_finetune(text, max_retries=4):
    """Call the fine-tuned model on Together serverless. Returns a dict
    with 'label', 'input_tokens', 'output_tokens'.
    """
    last_exc = None
    for attempt in range(max_retries):
        try:
            # YOUR CODE: call together_client.chat.completions.create with
            # model=FINETUNE_MODEL_ID, max_tokens=8, temperature=0, and one
            # user message containing the ticket text. Bind the return value
            # to `resp`.

            label = resp.choices[0].message.content.strip().lower()
            usage = resp.usage
            return {
                "label": label,
                "input_tokens": getattr(usage, "prompt_tokens", 0) or 0,
                "output_tokens": getattr(usage, "completion_tokens", 0) or 0,
            }
        except Exception as exc:
            last_exc = exc
            sleep_s = min(60, 2 ** attempt * 5)
            print(f"  attempt {attempt + 1}/{max_retries} failed ({exc!r}); sleeping {sleep_s}s")
            time.sleep(sleep_s)
    raise RuntimeError(f"fine-tune inference failed after {max_retries} retries: {last_exc!r}")


print("Warming up the serverless endpoint (cold-start can take 30 to 60s).")
warmup_text = "I was charged twice for my subscription this month."
warmup = classify_with_finetune(warmup_text)
print(f"  prediction: {warmup['label']!r}; tokens: {warmup['input_tokens']} in / {warmup['output_tokens']} out")


In [ ]:
# NBVAL_SKIP
# Checkpoint 3: the fine-tuned model serves inference. Validate by calling
# the model with a canonical example and confirming a non-empty string back
# (the accuracy check itself is CP4).


def cp3_validator(session):
    model_id = CLEANUP_MANIFEST[1].id
    if not model_id or model_id.startswith("placeholder"):
        return CheckpointResult(
            "fail",
            error_reason="no fine-tuned model id recorded; complete the fine-tune first",
        )
    try:
        resp = together_client.chat.completions.create(
            model=model_id,
            max_tokens=8,
            temperature=0,
            messages=[{
                "role": "user",
                "content": "Where is my package? Tracking has not updated in 4 days.",
            }],
        )
    except Exception as exc:
        return CheckpointResult(
            "error",
            error_reason=(
                f"serverless inference call raised {exc!r}. "
                f"First call may cold-start in 30 to 60s; retry the warmup cell with backoff."
            ),
        )
    content = resp.choices[0].message.content
    if not isinstance(content, str) or not content.strip():
        return CheckpointResult(
            "fail",
            error_reason=f"serverless endpoint returned empty content: {content!r}",
        )
    return CheckpointResult("pass")


check(3, cp3_validator)


<details><summary>Hint 1 (nudge)</summary>

A first-call timeout is almost never a real failure; Together's serverless cold-start for a freshly fine-tuned model can take a minute. Retry with backoff before you assume the model is broken.

</details>

<details><summary>Hint 2 (direction)</summary>

The Together chat-completions API is OpenAI-shaped:

```
together_client.chat.completions.create(
    model=<model_id>,
    max_tokens=...,
    temperature=...,
    messages=[{"role": "user", "content": "..."}],
)
```

The response shape mirrors OpenAI's too: `resp.choices[0].message.content` for the text, `resp.usage.prompt_tokens` / `resp.usage.completion_tokens` for token counts. Use `temperature=0` for classification; you want determinism.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Full working classify call:

```python
resp = together_client.chat.completions.create(
    model=FINETUNE_MODEL_ID,
    max_tokens=8,
    temperature=0,
    messages=[{"role": "user", "content": text}],
)
```

</details>

**Wallet check.** Fine-tune warmup plus a few cold-start retries: under one cent. Running total: about $7.40.


## Task 4: Benchmark accuracy on the 500-example test set

Run BOTH classifiers (fine-tune + Haiku baseline) on the 500-example test set, capture per-row predictions and token usage, write `/content/benchmark_results.csv`. Compute accuracy for each. The checkpoint asserts the fine-tune is within 2 percentage points of Haiku (or beats Haiku by any amount).

500 rows x 2 models = 1000 inference calls. Run them sequentially with brief pauses; rate-limit safety beats wall-clock for 1K calls. Total: about 8 minutes wall-clock, total cost a few cents.


In [ ]:
# NBVAL_SKIP
# Task 4: benchmark fine-tune vs Haiku on the 500-example test set.

# Load the test set from S3.
test_obj = s3_client.get_object(Bucket=S3_BUCKET, Key=TEST_KEY)
test_rows = [json.loads(line) for line in test_obj["Body"].read().decode("utf-8").splitlines() if line.strip()]
print(f"Loaded {len(test_rows)} test rows from s3://{S3_BUCKET}/{TEST_KEY}.")


HAIKU_PROMPT_TEMPLATE = (
    "Classify the following customer support ticket into exactly ONE of these "
    "five categories: billing, account, technical, shipping, refund. "
    "Reply with ONLY the category word, no punctuation, no preamble.\n\n"
    "Ticket: {ticket!r}"
)


def classify_with_haiku(text):
    """Call Claude Haiku 4.5 with the prompt template. Returns label + tokens."""
    prompt = HAIKU_PROMPT_TEMPLATE.format(ticket=text)
    # YOUR CODE: call anthropic_client.messages.create with model
    # ANTHROPIC_HAIKU_MODEL, max_tokens=8, and the prompt above as a single
    # user message. Bind the return value to `resp`.

    label = resp.content[0].text.strip().lower()
    return {
        "label": label,
        "input_tokens": resp.usage.input_tokens,
        "output_tokens": resp.usage.output_tokens,
    }


# Run both classifiers across the test set. Sequential and slow on purpose;
# 1K calls finish in roughly 8 minutes and rate limits stay calm.
records = []
last_print = time.time()
for i, row in enumerate(test_rows):
    ticket = row["messages"][0]["content"]
    gold = row["messages"][1]["content"]
    try:
        haiku = classify_with_haiku(ticket)
    except Exception as exc:
        print(f"  haiku failure on row {i} ({exc!r}); skipping")
        continue
    try:
        finetune = classify_with_finetune(ticket)
    except Exception as exc:
        print(f"  fine-tune failure on row {i} ({exc!r}); skipping")
        continue
    records.append({
        "row_idx": i,
        "ticket": ticket,
        "gold_label": gold,
        "haiku_pred": haiku["label"],
        "haiku_input_tokens": haiku["input_tokens"],
        "haiku_output_tokens": haiku["output_tokens"],
        "finetune_pred": finetune["label"],
        "finetune_input_tokens": finetune["input_tokens"],
        "finetune_output_tokens": finetune["output_tokens"],
    })
    if time.time() - last_print > 15:
        print(f"  {i + 1}/{len(test_rows)} rows benchmarked")
        last_print = time.time()


df = pd.DataFrame(records)
df.to_csv(BENCHMARK_CSV_PATH, index=False)
print(f"\nWrote {len(df)} rows to {BENCHMARK_CSV_PATH}.")

haiku_acc = (df["haiku_pred"] == df["gold_label"]).mean()
finetune_acc = (df["finetune_pred"] == df["gold_label"]).mean()
print(f"\nAccuracy snapshot:")
print(f"  Haiku baseline   : {haiku_acc:.3f}")
print(f"  Fine-tuned Llama : {finetune_acc:.3f}")
print(f"  Gap              : {abs(haiku_acc - finetune_acc):.3f} ({'fine-tune' if finetune_acc > haiku_acc else 'haiku'} ahead)")


In [ ]:
# NBVAL_SKIP
# Checkpoint 4: benchmark accuracy within 2 percentage points of Haiku, or
# fine-tune beats Haiku. Validator reads the CSV (the source of truth for
# both metrics) and recomputes both numbers.


def cp4_validator(session):
    if not os.path.exists(BENCHMARK_CSV_PATH):
        return CheckpointResult(
            "fail",
            error_reason=f"{BENCHMARK_CSV_PATH} not found; run Task 4 first",
        )
    df = pd.read_csv(BENCHMARK_CSV_PATH)
    required_cols = {
        "gold_label", "haiku_pred", "finetune_pred",
        "haiku_input_tokens", "haiku_output_tokens",
        "finetune_input_tokens", "finetune_output_tokens",
    }
    missing = required_cols - set(df.columns)
    if missing:
        return CheckpointResult(
            "fail",
            error_reason=f"{BENCHMARK_CSV_PATH} missing required columns: {sorted(missing)}",
        )
    if len(df) < TARGET_TEST_ROWS * 0.9:
        return CheckpointResult(
            "fail",
            error_reason=f"only {len(df)} rows in benchmark CSV; expected at least {int(TARGET_TEST_ROWS * 0.9)}",
        )

    haiku_acc = (df["haiku_pred"] == df["gold_label"]).mean()
    finetune_acc = (df["finetune_pred"] == df["gold_label"]).mean()

    # Pass if abs(haiku - finetune) <= 0.02 OR finetune >= haiku - 0.02.
    if not (abs(haiku_acc - finetune_acc) <= 0.02 or finetune_acc >= haiku_acc - 0.02):
        return CheckpointResult(
            "fail",
            error_reason=(
                f"accuracy gap too wide: haiku={haiku_acc:.3f}, finetune={finetune_acc:.3f}. "
                f"Fine-tune must be within 2 points of Haiku. Inspect per-class accuracy in the CSV."
            ),
        )
    return CheckpointResult("pass")


check(4, cp4_validator)


<details><summary>Hint 1 (nudge)</summary>

If your accuracy gap is wider than 2 points, do not blame the model first. Inspect per-class accuracy. Group the CSV by `gold_label` and compare `haiku_pred == gold_label` against `finetune_pred == gold_label` per class. One weak class drags the overall number; that is usually a synthetic-data balance or prompt-clarity issue, not a fine-tuning issue.

</details>

<details><summary>Hint 2 (direction)</summary>

The Haiku prompt MUST constrain output to a single category word with no preamble. If Haiku replies "This sounds like a billing issue." instead of "billing", the `==` accuracy check counts it wrong. `max_tokens=8` plus the explicit "reply with ONLY the category word" instruction keeps Haiku in line.

For the fine-tune, the model was trained on `{"role": "assistant", "content": "<label>"}` directly, so it learns to output just the label and almost never adds a preamble. If your fine-tune outputs are noisy, run a sanity check on 5 rows before walking the full 500.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Full working Haiku classify call:

```python
resp = anthropic_client.messages.create(
    model=ANTHROPIC_HAIKU_MODEL,
    max_tokens=8,
    messages=[{"role": "user", "content": prompt}],
)
```

Quick per-class diagnostic:

```python
import pandas as pd
df = pd.read_csv(BENCHMARK_CSV_PATH)
for lab in df["gold_label"].unique():
    sub = df[df["gold_label"] == lab]
    h = (sub["haiku_pred"] == sub["gold_label"]).mean()
    f = (sub["finetune_pred"] == sub["gold_label"]).mean()
    print(f"{lab:>10}  haiku={h:.3f}  finetune={f:.3f}  n={len(sub)}")
```

</details>

**Wallet check.** 500 Haiku calls plus 500 fine-tune calls. Roughly 60K Haiku tokens in + 4K Haiku tokens out (about 6 cents); 60K Together tokens in + 4K Together tokens out (about 1.3 cents). Running total: about $7.50.


## Task 5: Compute cost-per-1K and confirm the fine-tune is at least 5x cheaper

Use the token-usage columns in the benchmark CSV. **Pricing is in dollars per 1M tokens, not per 1K**; do not confuse units. Haiku: $0.80 input / $4.00 output per 1M. Together Llama serverless: $0.20 input / $0.20 output per 1M.

Cost per 1K classifications =
`(avg_input_tokens * input_rate + avg_output_tokens * output_rate) * 1000 / 1_000_000`.

If the ratio Haiku / fine-tune is at least 5x, CP5 passes. With the rates above and similar token counts, the typical ratio lands around 7x to 9x.


In [ ]:
# NBVAL_SKIP
# Task 5: compute per-1K cost for each model from the CSV.

df = pd.read_csv(BENCHMARK_CSV_PATH)


def _cost_per_1k(input_total_tokens, output_total_tokens, n_calls,
                 input_rate_per_m, output_rate_per_m):
    """Return USD per 1K calls for one model."""
    # YOUR CODE: compute the average input + output tokens per call from the
    # totals and the call count, then convert to dollars per 1000 calls using
    # the per-million-token rates. Bind the answer to `cost_per_1k`.

    return cost_per_1k


haiku_cost_per_1k = _cost_per_1k(
    df["haiku_input_tokens"].sum(),
    df["haiku_output_tokens"].sum(),
    len(df),
    HAIKU_INPUT_USD_PER_M,
    HAIKU_OUTPUT_USD_PER_M,
)
finetune_cost_per_1k = _cost_per_1k(
    df["finetune_input_tokens"].sum(),
    df["finetune_output_tokens"].sum(),
    len(df),
    TOGETHER_LLAMA_INPUT_USD_PER_M,
    TOGETHER_LLAMA_OUTPUT_USD_PER_M,
)

ratio = haiku_cost_per_1k / finetune_cost_per_1k if finetune_cost_per_1k > 0 else float("inf")

print("Cost per 1K classifications:")
print(f"  Haiku baseline   : ${haiku_cost_per_1k:.4f}")
print(f"  Fine-tuned Llama : ${finetune_cost_per_1k:.4f}")
print(f"  Ratio (Haiku / fine-tune): {ratio:.2f}x")

# Persist the summary as a single-row companion CSV next to the per-row file
# so an inspector can read the exact same numbers without recomputing.
summary_path = "/content/benchmark_cost_summary.csv"
pd.DataFrame([{
    "haiku_cost_per_1k": haiku_cost_per_1k,
    "finetune_cost_per_1k": finetune_cost_per_1k,
    "ratio": ratio,
}]).to_csv(summary_path, index=False)
print(f"\nWrote cost summary to {summary_path}.")


In [ ]:
# NBVAL_SKIP
# Checkpoint 5: cost-per-1K of Haiku divided by cost-per-1K of the fine-tune
# is at least 5x. Validator recomputes from the per-row CSV so a manual
# summary tweak cannot cheat the check.


def cp5_validator(session):
    if not os.path.exists(BENCHMARK_CSV_PATH):
        return CheckpointResult(
            "fail",
            error_reason=f"{BENCHMARK_CSV_PATH} not found; run Task 4 first",
        )
    df = pd.read_csv(BENCHMARK_CSV_PATH)
    if len(df) < TARGET_TEST_ROWS * 0.9:
        return CheckpointResult(
            "fail",
            error_reason=f"only {len(df)} rows in benchmark CSV; expected at least {int(TARGET_TEST_ROWS * 0.9)}",
        )

    n = len(df)
    haiku_avg_in = df["haiku_input_tokens"].sum() / n
    haiku_avg_out = df["haiku_output_tokens"].sum() / n
    finetune_avg_in = df["finetune_input_tokens"].sum() / n
    finetune_avg_out = df["finetune_output_tokens"].sum() / n

    haiku_per_1k = (
        haiku_avg_in * HAIKU_INPUT_USD_PER_M
        + haiku_avg_out * HAIKU_OUTPUT_USD_PER_M
    ) * 1000 / 1_000_000
    finetune_per_1k = (
        finetune_avg_in * TOGETHER_LLAMA_INPUT_USD_PER_M
        + finetune_avg_out * TOGETHER_LLAMA_OUTPUT_USD_PER_M
    ) * 1000 / 1_000_000

    if finetune_per_1k <= 0:
        return CheckpointResult(
            "fail",
            error_reason=f"fine-tune cost-per-1K is {finetune_per_1k:.4f}; token counts may be missing",
        )
    ratio = haiku_per_1k / finetune_per_1k
    if ratio < 5.0:
        return CheckpointResult(
            "fail",
            error_reason=(
                f"cost ratio is {ratio:.2f}x (need at least 5x). "
                f"Haiku=${haiku_per_1k:.4f}/1K, fine-tune=${finetune_per_1k:.4f}/1K. "
                f"Double-check you used Haiku's $0.80/$4.00 per 1M and Together's $0.20/$0.20 per 1M."
            ),
        )
    return CheckpointResult("pass")


check(5, cp5_validator)


<details><summary>Hint 1 (nudge)</summary>

If your ratio is below 5x, you almost certainly mixed pricing units. Haiku is billed per-million tokens with DIFFERENT input and output rates ($0.80 input, $4.00 output). Together's Llama serverless is also per-million with the same input/output rate ($0.20 / $0.20). Do not multiply by 1000 in the wrong place.

</details>

<details><summary>Hint 2 (direction)</summary>

The cost-per-1K formula:

```
cost_per_1k = (avg_input_tokens * input_rate_per_M + avg_output_tokens * output_rate_per_M) * 1000 / 1_000_000
```

where `avg_input_tokens = total_input_tokens / n_calls`. The `* 1000 / 1_000_000` converts from "rate per million tokens, per single call" to "dollars per 1000 calls".

Sanity check: a Haiku classification call uses about 120 input + 4 output tokens. Cost per 1K is roughly (120 * 0.80 + 4 * 4.00) * 1000 / 1_000_000 = $0.112. Together at the same token counts: (120 * 0.20 + 4 * 0.20) * 1000 / 1_000_000 = $0.0248. Ratio: 4.5x to 9x depending on how output-heavy your prompt is.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Full working cost helper:

```python
def _cost_per_1k(input_total_tokens, output_total_tokens, n_calls,
                 input_rate_per_m, output_rate_per_m):
    avg_in = input_total_tokens / n_calls
    avg_out = output_total_tokens / n_calls
    cost_per_call = (avg_in * input_rate_per_m + avg_out * output_rate_per_m) / 1_000_000
    cost_per_1k = cost_per_call * 1000
    return cost_per_1k
```

</details>

**Wallet check.** No new spend on this task; pure CSV math. Running total: about $7.50. The fine-tune itself is most of the spend.


## Cleanup

Cancel the Together fine-tune job if still running (no-op in the success path), delete the fine-tuned model, empty + delete the S3 bucket, remove the local benchmark CSV. The verification scan re-queries Together and S3 to confirm everything is gone; the orphan scan covers any prior session's leftovers.


In [ ]:
# NBVAL_SKIP
# Cleanup. Critical first (Together fine-tune job), then standard.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id and res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_total = sum(1 for r in CLEANUP_MANIFEST if r.critical)
standard_total = len(CLEANUP_MANIFEST) - critical_total
critical_destroyed = critical_total - sum(1 for r in CLEANUP_MANIFEST if r.critical and r.id in failed_ids)
standard_destroyed = standard_total - sum(1 for r in CLEANUP_MANIFEST if not r.critical and r.id in failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Together AI account or AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)


**Session total: about $8.00.** The fine-tune was $5 to $8 (Together QLoRA on 5K rows, 3 epochs, Llama 3.1 8B). Synthetic data generation with Sonnet was about $0.90. Haiku baseline benchmarking was about 6 cents. Fine-tune serverless inference for the 500-row benchmark was about 1.3 cents. S3 storage was free-tier. Cleanup deleted the fine-tuned model and dropped the S3 bucket so no idle billing remains. Check your Together AI dashboard and AWS Billing console in 24 hours to confirm.

Eight bucks. Fine-tune done. Together bill gone. S3 bucket gone. No idle billing.


## Reflection

These are not graded. They are for you.

1. The fine-tune cost $6.50 and the accuracy gap is 2 points (fine-tune at 87%, Haiku at 89%, say). At what monthly inference volume does the fine-tune break even on payback? Show the math. (Hint: `monthly_savings = volume_per_1K * (haiku_per_1K - finetune_per_1K)`. Solve for the volume where `monthly_savings >= 6.50`.)

2. The team is considering keeping Haiku as the fallback when the fine-tune's confidence is below a threshold. What is one risk of this pattern beyond cost? (Hint: think about distribution drift over time; what happens if the fine-tune's confidence calibration degrades on a new ticket category that did not appear in the training set?)

## Exam-Style Practice

**Question 1 (cost-vs-accuracy trade-off):**

A team has a classification task at 91% accuracy on Haiku for $5K/month. A fine-tune of Llama 3.1 8B hits 88% for $1K/month. The product owner needs 90% accuracy. Which is the right call?

A. Ship the fine-tune; 88% is close enough.
B. Stay on Haiku; 91% beats 90%.
C. Use the fine-tune as primary, fall back to Haiku when the fine-tune's confidence is below a threshold.
D. Re-train on more data and hope for an extra 2 points.

<details><summary>Show answer</summary>

**Correct: C.**

- A is wrong: 88% misses the 90% accuracy bar. "Close enough" violates the explicit product requirement.
- B is wrong: stays at $5K/month and ignores the cost lever entirely. The product owner specified 90%, not 91%.
- C is correct: the fallback pattern reclaims the 2-point accuracy gap at a fraction of the cost difference. In practice a calibrated confidence threshold sends roughly 10 to 20% of traffic to Haiku, which typically pulls effective accuracy to 90%+ while cutting cost by 60 to 80%.
- D is wrong: more data MIGHT help, but the answer is speculative and ignores the working architecture in C. In a senior interview, "throw more data at it" without a calibrated cost/accuracy plan is a red flag.

</details>

**Question 2 (fine-tune data shape):**

A team's first Together fine-tune job rejects at submit with `Invalid training data format`. They are uploading a JSONL file where each line looks like:

```
{"text": "Customer ticket: I was charged twice.\nLabel: billing"}
```

Which is the correct fix?

A. Switch to JSON (one object, not one-per-line).
B. Move to OpenAI's fine-tune API; Together does not support this task.
C. Reshape each line as `{"messages": [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}` to match Together's chat-completions schema.
D. Compress the JSONL with gzip before uploading.

<details><summary>Show answer</summary>

**Correct: C.**

- A is wrong: Together's fine-tune API expects JSONL (one JSON object per line), not a single JSON document.
- B is wrong: Together fully supports text classification fine-tunes on Llama 3.1 8B; the rejection is a schema issue.
- C is correct: the `{"messages": [user, assistant]}` chat-completions schema is the documented input format for Together's QLoRA fine-tunes on chat-instruct base models. The legacy `{"text": "<prompt><sep><completion>"}` format is rejected.
- D is wrong: gzip compression is unrelated to a format-rejection error.

</details>

**Question 3 (re-entry safety):**

A student starts a 50-minute Together fine-tune job, then the Colab kernel disconnects 20 minutes in. What is true?

A. The fine-tune is cancelled and the student is refunded the partial run.
B. The fine-tune is cancelled and the student is billed for the full run anyway.
C. The fine-tune keeps running on Together's side; on reconnect, the student adopts the in-flight job by suffix and resumes polling with no double charge.
D. The fine-tune duplicates: a new job starts on each subsequent reconnect.

<details><summary>Show answer</summary>

**Correct: C.**

- A is wrong: a client disconnect is not a server-side cancel. Together does not have a refund flow for partial fine-tunes on disconnect.
- B is wrong: the fine-tune is NOT cancelled by a client disconnect.
- C is correct: Together's fine-tune jobs run server-side and continue regardless of client state. The lab's `suffix=TOGETHER_SUFFIX` tag lets the re-entry scan in setup find the in-flight job and adopt it; the student polls until completion with no new spend.
- D is wrong: only if the student re-submits via a fresh `fine_tuning.create` call. The lab's setup explicitly prevents that by checking for an existing job with the lab's suffix before submitting.

</details>
